# LM5176 EN/UVLO Drop-Out Lockout Latch — Verified Analysis

Discrete BJT latch that pulls the **EN/UVLO** pin of an LM5176 4-switch
buck-boost controller **low** when `VIN` drops out, and *holds* the IC off
(ignoring any `VIN` return) until `VOUT` has decayed. This prevents the
pre-bias / back-feed restart problem.

This notebook is a cleaned, **schematic-consistent** rewrite of the original
analysis. Everything here is keyed to the reference designators in
`en_uvlo_lockout.asc` (parsed programmatically, not hand-traced).

> **Scope / honesty note.** The closed-form inequalities below are the design
> intent. The DC sweep at the end uses a *generic* Ebers-Moll BJT model
> (not the exact 2N3904/2N3906 SPICE models) and is a sanity check only.
> Absolute trip voltages — especially the RESET point — **must be confirmed
> in LTSpice** with the real device models.

## 1. Verified topology & device roles

Parsed directly from the `.asc` netlist (union-find on pin coordinates).
`VB3` is the **latch core node**: the shared base of Q3 and Q4.

| Ref | Type | Pins (B / C / E) | Role |
|-----|------|------------------|------|
| **Q1** | NPN 2N3904 | B=VB1, C→R4→VB3, E=GND | **VIN-high sensor / reset pulldown.** Divider R1/R2 sets base. Off when VIN low. |
| **Q2** | PNP 2N3906 | B=VB2P, C→R5→VB3, E=VOUT | **Feedback injector.** Sources from VOUT into VB3 to hold the latch. |
| **Q3** | NPN 2N3904 | B=VB3, C→R7→EN_UVLO, E=GND | **Latch output.** Pulls EN/UVLO low → disables IC. |
| **Q4** | NPN 2N3904 | B=VB3, C→R10→VB2P, E=GND | **Feedback driver.** Pulls VB2P down → turns Q2 on. |

**Resistors**

| Ref | Value | Net A → Net B | Function |
|-----|-------|---------------|----------|
| R1  | 5 k   | VIN → VB1   | VIN-sense divider (top) |
| R2  | 1 k   | VB1 → GND   | VIN-sense divider (bottom) → VB1 = VIN/6 |
| R3  | 100 k | VOUT → VB3  | **SET** pull-up: “VOUT still high” senses dropout |
| R4  | 1 k   | VB3 → Q1.C  | **RESET** pull-down when VIN high (via Q1) |
| R5  | 4 k   | Q2.C → VB3  | **HOLD** regenerative feedback pull-up from VOUT |
| R6  | 10 k  | VB3 → GND   | Bleed / noise immunity on the core node |
| R7  | 1 k   | Q3.C → EN_UVLO | Series R for the EN pulldown |
| R8  | 300 k | VIN → EN_UVLO | EN/UVLO divider (top) |
| R9  | 100 k | EN_UVLO → GND | EN/UVLO divider (bottom) → ratio 0.25 |
| R10 | 100 k | VB2P → Q4.C | Q4 collector load (pulls VB2P down) |
| R11 | 10 k  | VOUT → VB2P | Keeps Q2 off (VB2P=VOUT) until Q4 pulls |

**Latch loop in words:** VIN drops → Q1 off → R3 (from still-high VOUT) pulls
VB3 up → Q3 on (EN low, IC off) **and** Q4 on → Q2 on → R5 feeds VB3 from VOUT
= regenerative HOLD. VIN returning turns Q1 back on (R4 fights the latch) but
feedback wins **while VOUT is high**. When VOUT decays, feedback collapses →
RESET → EN recovers → IC restarts.

> **Designator warning.** In the *original* notebook `r1=100k / r2=300k` were
> called the “EN/UVLO divider.” That is wrong: in the schematic the UVLO
> divider is **R8/R9 = 300k/100k**, and **R1/R2 = 5k/1k** is the VIN-sense
> divider for Q1. The original Q1–Q4 indices were also shuffled. All names
> below follow the **schematic**.

## 2. Verified LM5176 EN/UVLO thresholds

From the TI LM5176 datasheet (Electrical Characteristics) and `constants.py`:

| Symbol | Meaning | Min | Typ | Max | Unit |
|--------|---------|-----|-----|-----|------|
| — | Shutdown (EN below this) | | | 0.40 | V |
| V_EN(OP) | Operating / PWM enable, rising | 1.17 | **1.22** | 1.29 | V |
| I_EN(STBY) | Standby current **sourced from** the pin | | **2.0** | 4.0 | µA |
| ΔI_HYS(OP) | Extra hysteresis current after enable | 2.15 | **3.15** | 4.25 | µA |

**Datasheet UVLO equation** (the EN pin *sources* current into the divider tap,
with R_UV2 = top = R8, R_UV1 = bottom = R9):

$$ V_{IN,uv} = V_{EN,op}\Bigl(1 + \frac{R_{UV2}}{R_{UV1}}\Bigr) - R_{UV2}\,I_{EN} $$

```python
v_in_uv = v_en_op * (1 + r_uv2 / r_uv1) - r_uv2 * i_en_stby
```

The sourced pin current makes the term **subtract**, lowering the trip. Before
enable only I_STBY flows; after enable an extra ΔI_HYS flows, so the falling
trip drops by a further R8·ΔI_HYS. With **R8/R9 = 300k/100k** (typ):

- **VIN rising enable:** 1.22·4 − 300k·2.0µA = **4.28 V**
- **VIN falling disable:** 1.22·4 − 300k·(2.0+3.15)µA = **3.34 V**
- **hysteresis window:** R8·ΔI_HYS = **0.95 V**

> **Correction to the earlier pass.** My first review used 4.88 V / 3.94 V —
> that **ignored the standby-current offset** (R8·I_STBY = 0.60 V). The datasheet
> equation you supplied shifts both trips down by that amount. Net effect: the
> 300k/100k divider actually lands the **turn-on at 4.28 V**, neatly inside the
> `DESIGN_TARGETS` window [V_ON_SAFE_FLOOR = 3.8 V, V_USB_MIN = 4.5 V]. The
> falling trip 3.34 V sits just under V_OFF_MIN = 3.6 V — see the reconciliation
> cell below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
#  SINGLE SOURCE OF TRUTH  (schematic-consistent)
# ============================================================

# --- Component values from en_uvlo_lockout.asc (ohms) ---
R1, R2  = 5e3,   1e3      # VIN-sense divider  -> VB1 = VIN * R2/(R1+R2) = VIN/6
R3      = 100e3           # SET pull-up  VOUT -> VB3
R4      = 1e3             # RESET pull-down VB3 -> Q1 collector
R5      = 4e3             # HOLD feedback  Q2 collector -> VB3
R6      = 10e3            # VB3 bleed to GND
R7      = 1e3             # EN pulldown series R (Q3 collector)
R8, R9  = 300e3, 100e3    # EN/UVLO divider: R8=top(VIN->EN), R9=bottom(EN->GND)
R10     = 100e3           # Q4 collector load -> VB2P
R11     = 10e3            # VOUT -> VB2P keep-off

# --- IC + design numbers: pull from constants.py if importable, else inline ---
# (constants.py needs component CSVs for the inductor/FET; the IC + target
#  numbers themselves are dependency-free, so we mirror them on fallback.)
try:
    from constants import IC_LM5176 as _IC, DESIGN_TARGETS as _DT
    V_EN_OP   = _IC.V_EN_NOM        # 1.22 V
    I_STBY    = _IC.I_STBY_NOM      # 2.0 uA, sourced from the pin
    I_HYS     = _IC.I_HYS_NOM       # 3.15 uA, extra after enable
    V_USB_MIN, V_ON_SAFE_FLOOR = _DT.V_USB_MIN, _DT.V_ON_SAFE_FLOOR
    V_OFF_MIN, HYS_MIN_GAP, MAX_DIVIDER_UW = _DT.V_OFF_MIN, _DT.HYS_MIN_GAP, _DT.MAX_DIVIDER_UW
    print('using values from constants.py')
except Exception as e:
    V_EN_OP, I_STBY, I_HYS = 1.22, 2.0e-6, 3.15e-6
    V_USB_MIN, V_ON_SAFE_FLOOR, V_OFF_MIN = 4.50, 3.8, 3.6
    HYS_MIN_GAP, MAX_DIVIDER_UW = 0.25, 500.0
    print('constants.py not importable here; using inline mirror of its values')

# --- LM5176 UVLO trip, datasheet equation (pin SOURCES current) ---
def vin_uvlo(R_top, R_bot, i_pin):
    return V_EN_OP * (1 + R_top / R_bot) - R_top * i_pin

VIN_RISE = vin_uvlo(R8, R9, I_STBY)            # before enable: I_STBY only
VIN_FALL = vin_uvlo(R8, R9, I_STBY + I_HYS)    # after enable: + hysteresis current
UVLO_RATIO = R9 / (R8 + R9)

# --- BJT first-order constants ---
VBE_ON   = 0.65      # turn-on / active base-emitter
VBE_SAT  = 0.75      # hard-sat base-emitter
VBE_OFF  = 0.50      # below this -> effectively off
VCE_SAT  = 0.10      # saturated collector-emitter
BETA_FORCED = 10     # forced-beta design margin (Ic/Ib must stay <= this)

# --- Operating envelope ---
VOUT_NOM       = 12.0    # regulated output
VOUT_RELEASE   = 2.0     # target: latch must still HOLD above this, may reset below
VIN_USB_MIN    = 4.5     # USB source can sag to here
VIN_USB_MAX    = 22.5    # PD source can rise to here

print(f'UVLO divider ratio        : {UVLO_RATIO:.3f}')
print(f'VIN rising  enable        : {VIN_RISE:.2f} V')
print(f'VIN falling disable       : {VIN_FALL:.2f} V')
print(f'hysteresis window         : {VIN_RISE-VIN_FALL:.2f} V')
print(f'VB1 = VIN * {R2/(R1+R2):.3f}  -> Q1 releases near VIN = '
      f'{VBE_ON*(R1+R2)/R2:.2f} V (VBE_ON) to {VBE_SAT*(R1+R2)/R2:.2f} V (VBE_SAT)')

## 3. Reconciliation against `DESIGN_TARGETS`

Check the realized 300k/100k divider against every target in `constants.py`,
and sweep the divider power across the input range.

In [ ]:
checks = [
    ('turn ON at/below V_USB_MIN', VIN_RISE <= V_USB_MIN,
     f'rising {VIN_RISE:.2f} V vs {V_USB_MIN} V'),
    ('do NOT turn on below V_ON_SAFE_FLOOR', VIN_RISE >= V_ON_SAFE_FLOOR,
     f'rising {VIN_RISE:.2f} V vs {V_ON_SAFE_FLOOR} V'),
    ('IC off below V_OFF_MIN', VIN_FALL <= V_OFF_MIN,
     f'falling {VIN_FALL:.2f} V vs {V_OFF_MIN} V (IC self-off; latch should grab above this)'),
    ('hysteresis >= HYS_MIN_GAP', (VIN_RISE-VIN_FALL) >= HYS_MIN_GAP,
     f'window {VIN_RISE-VIN_FALL:.2f} V vs {HYS_MIN_GAP} V'),
]
print('TARGET RECONCILIATION (R8/R9 = 300k/100k)')
for name, ok, detail in checks:
    print(f'  [{"PASS" if ok else "FAIL"}] {name:38s} {detail}')

# --- divider power vs VIN against MAX_DIVIDER_UW ---
vin = np.linspace(5, 22.5, 200)
p_uw = vin**2 / (R8 + R9) * 1e6
v_break = np.sqrt(MAX_DIVIDER_UW*1e-6 * (R8+R9))
print(f'\n  divider power: {5**2/(R8+R9)*1e6:.0f} uW @5V -> {22.5**2/(R8+R9)*1e6:.0f} uW @22.5V')
print(f'  budget {MAX_DIVIDER_UW:.0f} uW exceeded above VIN = {v_break:.1f} V  '
      f'-> FAIL across the PD range (15-20 V)')

fig, ax = plt.subplots(figsize=(8,4.5))
ax.plot(vin, p_uw, label='R8+R9 divider power')
ax.axhline(MAX_DIVIDER_UW, color='red', ls='--', label=f'budget {MAX_DIVIDER_UW:.0f} uW')
ax.axvline(v_break, color='gray', ls=':', label=f'break-even {v_break:.1f} V')
ax.fill_between(vin, MAX_DIVIDER_UW, p_uw, where=p_uw>MAX_DIVIDER_UW, color='red', alpha=0.15)
ax.set_xlabel('VIN (V)'); ax.set_ylabel('divider power (uW)')
ax.set_title('EN/UVLO divider power vs budget'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 4. The four operating regions

The design must satisfy one inequality in each region. All node names refer
to the schematic; `VB3` is the latch-core (Q3/Q4 base).

**(a) Startup / steady-state — must NOT latch (VIN high, VOUT high).**
Q1 is on, so its collector sits near GND and R4 (1k) pulls VB3 down hard.
R4 must dominate the R3 (100k) pull-up from VOUT so VB3 < VBE_OFF and Q3/Q4
stay off:
$$ V_{B3} \approx V_{OUT}\frac{R4}{R3+R4} + V_{CE,sat}\frac{R3}{R3+R4} < V_{BE,off} $$

**(b) SET on VIN dropout (VIN falling, VOUT still high).**
When VIN falls enough that Q1 turns off, R4's pull-down is released and R3
pulls VB3 up to turn Q3/Q4 on. Q1 turns off near:
$$ V_{IN,set} \approx V_{BE,on}\,\frac{R1+R2}{R2} $$
With R1/R2 = 5k/1k this is ~3.9–4.2 V (Q1 conduction edge), and the DC solver
shows the latch fully engaged (EN < 0.4 V) by ~3.5 V. Both sit **above** the
corrected IC falling trip of **3.34 V**, so the latch grabs control before the
IC self-disables — the original design intent. Margin is thin (~0.2 V) and the
full-engage point (~3.5 V) is marginally below V_OFF_MIN (3.6 V); nudging
R2/(R1+R2) up buys margin if you want it.

**(c) HOLD with VIN back high (VIN high, VOUT high).**
Q1 is on again and fights via R4, but the feedback (Q2 sourcing from VOUT
through R5) into VB3 must win. Feedback current into VB3 must exceed the R4
sink plus the base current Q3/Q4 need:
$$ \frac{V_{OUT}-V_{EB,Q2}-V_{CE,sat}}{R5} \;>\; \frac{V_{B3}-V_{CE,sat}}{R4} + I_{B,Q3}+I_{B,Q4} $$

**(d) RESET when VOUT falls (VIN high, VOUT low).**
As VOUT decays, both the R3 pull-up and the R5 feedback weaken. Below the
release point the R4 sink (and R6 bleed) win, VB3 drops below VBE_OFF, Q3/Q4
turn off, EN recovers:
$$ \text{feedback}(V_{OUT}) \;<\; \frac{V_{B3}-V_{CE,sat}}{R4} + \frac{V_{B3}}{R6} \quad\text{for } V_{OUT}<V_{release} $$

In [ ]:
# ============================================================
#  (b) SET trip: VIN-sense divider R1/R2 driving Q1
# ============================================================
# Q1 turns off (releasing the R4 reset pull-down) when VB1 = VIN*R2/(R1+R2)
# falls through the VBE turn-on band. The SET trip is where that happens.

ratio_12 = R2 / (R1 + R2)
vin_set_on  = VBE_ON  / ratio_12     # VIN where Q1 just conducts
vin_set_off = VBE_OFF / ratio_12     # VIN where Q1 fully off

print(f'R1/R2 = {R1:.0f}/{R2:.0f},  divider ratio = {ratio_12:.3f}')
print(f'Q1 conduction edge   VIN ~ {vin_set_on:.2f} V')
print(f'Q1 fully-off edge    VIN ~ {vin_set_off:.2f} V')
print(f'IC falling threshold VIN ~ {VIN_FALL:.2f} V (typ)')

# Sweep VIN, show VB1 vs the VBE band and the IC falling threshold
vin = np.linspace(0, 8, 400)
vb1 = vin * ratio_12

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(vin, vb1, label='VB1 = VIN · R2/(R1+R2)')
ax.axhspan(VBE_OFF, VBE_ON, color='orange', alpha=0.2, label='Q1 turn-off band')
ax.axvline(VIN_FALL, color='red', ls='--', label=f'IC falling {VIN_FALL:.2f} V')
ax.axvspan(vin_set_off, vin_set_on, color='green', alpha=0.15, label='SET window')
ax.set_xlabel('VIN (V)'); ax.set_ylabel('VB1 (V)')
ax.set_title('SET trip: VIN-sense divider R1/R2')
ax.legend(); ax.grid(alpha=0.3)
plt.show()

# The latch engages somewhere inside the Q1 turn-off band [off-edge, on-edge].
# The DC solver (later) pins full engagement at ~3.5 V. Compare that band to the trip.
print(f'\nQ1 turn-off band [{vin_set_off:.2f}, {vin_set_on:.2f}] V straddles '
      f'the IC falling trip {VIN_FALL:.2f} V.')
if vin_set_on > VIN_FALL:
    print('  DC solver shows full engage ~3.5 V (above the trip) -> latch grabs first. OK.')
    print(f'  Tight spot: full-engage ~3.5 V is marginally below V_OFF_MIN {V_OFF_MIN} V;')
    print('  raise R2/(R1+R2) (e.g. 4.7k/1.2k) if you want guaranteed margin.')
else:
    print('  Whole band is below the trip -> IC self-disables first. Raise R2/(R1+R2).')

In [ ]:
# ============================================================
#  (a)+(d) Latch-core design space:  R3 (SET pull-up) vs R4 (RESET pull-down)
# ============================================================
# Region (a): VIN & VOUT high, Q1 ON. VB3 must stay BELOW VBE_OFF so Q3/Q4 off.
#   VB3 ~ VOUT*R4/(R3+R4) + VCE_SAT*R3/(R3+R4)
# Region (b): after Q1 releases, R3 from VOUT must pull VB3 ABOVE VBE_ON.

r = np.logspace(2, 6, 400)          # 100 ohm .. 1 Mohm
R3g, R4g = np.meshgrid(r, r)

# (a) no spurious latch while Q1 on (collector ~ VCE_SAT)
vb3_steady = VOUT_NOM * R4g/(R3g+R4g) + VCE_SAT * R3g/(R3g+R4g)
c_no_latch = vb3_steady < VBE_OFF

# (b) once Q1 off, R3 from VOUT must drive VB3 high enough (R6 bleed to GND).
#     VB3 ~ VOUT*R6/(R3+R6) must exceed VBE_ON
vb3_set = VOUT_NOM * R6/(R3g + R6)
c_can_set = vb3_set > VBE_ON

valid = c_no_latch & c_can_set

fig, ax = plt.subplots(figsize=(8,6))
ax.contourf(R3g, R4g, valid, levels=[0.5,1.5], colors=['#2ca02c'], alpha=0.3)
ax.contour(R3g, R4g, vb3_steady, levels=[VBE_OFF], colors='red')
ax.contour(R3g, R4g, vb3_set,    levels=[VBE_ON], colors='blue')
ax.plot(R3, np.full_like(R3, R4), alpha=0)  # keep autoscale
ax.scatter([R3],[R4], color='k', zorder=5, label=f'design R3={R3/1e3:.0f}k, R4={R4/1e3:.0f}k')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('R3  (SET pull-up, Ω)'); ax.set_ylabel('R4  (RESET pull-down, Ω)')
ax.set_title('Latch-core space: green = no false latch AND can set')
ax.text(0.02,0.02,'red  = VB3(steady)=VBE_off (must be left/above)\n'
        'blue = VB3(set)=VBE_on (must be left)', transform=ax.transAxes,
        fontsize=8, va='bottom')
ax.legend(loc='upper right'); ax.grid(alpha=0.2, which='both')
plt.show()

In [ ]:
# ============================================================
#  (c) HOLD design space:  R5 (feedback) vs R6 (bleed)
# ============================================================
# Feedback current sourced into VB3 from VOUT (via Q2 + R5) must beat the R4
# sink (Q1 on) + R6 bleed + base current of Q3/Q4, evaluated at the WORST case
# (VOUT down near release, VIN up at max so Q1 sinks hardest).

r = np.logspace(2.5, 6, 400)
R5g, R6g = np.meshgrid(r, r)

# base current the pair needs (forced-beta), Q3 collector load = R7 to EN/VOUT-ish
ib_pair = (VOUT_NOM / R7) / BETA_FORCED + (VIN_USB_MAX / R2) / BETA_FORCED

# worst-case feedback when VOUT is near release
i_fb_worst = (VOUT_RELEASE - VBE_SAT - VCE_SAT) / R5g
# sinks at VB3 ~ VBE_SAT with Q1 collector near GND
i_sink = (VBE_SAT - VCE_SAT) / R4 + VBE_SAT / R6g + ib_pair
c_hold = i_fb_worst > i_sink

fig, ax = plt.subplots(figsize=(8,6))
ax.contourf(R5g, R6g, c_hold, levels=[0.5,1.5], colors=['#1f77b4'], alpha=0.3)
ax.contour(R5g, R6g, i_fb_worst - i_sink, levels=[0], colors='blue')
ax.scatter([R5],[R6], color='k', zorder=5, label=f'design R5={R5/1e3:.0f}k, R6={R6/1e3:.0f}k')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('R5  (feedback, Ω)'); ax.set_ylabel('R6  (bleed, Ω)')
ax.set_title(f'HOLD space (blue): feedback beats sink at VOUT={VOUT_RELEASE} V')
ax.legend(loc='upper right'); ax.grid(alpha=0.2, which='both')
plt.show()

print(f'base-current demand of Q3/Q4 pair (forced-beta {BETA_FORCED}): {ib_pair*1e6:.1f} uA')

## 5. DC verification (generic Ebers-Moll) — sanity check only

This solves the 8-node circuit at DC with a *generic* BJT model. It confirms
the **qualitative** behavior (SET / HOLD / RESET) and the all-important HOLD
margin. **Do not trust absolute trip voltages from this** — confirm in LTSpice
with the real 2N3904 / 2N3906 models.

Caveat already known: cold solves at high VIN/VOUT can diverge, so each sweep
is **continued** from a benign starting point.

In [ ]:
from scipy.optimize import fsolve

# generic Ebers-Moll (NOT the exact 2N3904/3906 models)
Vt, Is, betaF, betaR = 0.02585, 6.7e-15, 150.0, 4.0
I_EN = 2e-6   # IC standby current drawn out of EN node

def _ex(v): return np.exp(np.clip(v/Vt, -40, 40)) - 1.0
def npn(Vb,Vc,Ve):
    f,rr = _ex(Vb-Ve), _ex(Vb-Vc)
    return Is*(f-rr) - (Is/betaR)*rr, (Is/betaF)*f + (Is/betaR)*rr
def pnp(Vb,Vc,Ve):
    ic,ib = npn(-Vb,-Vc,-Ve); return -ic,-ib

def residuals(x, VIN, VOUT):
    VB1,VB3,VB2P,nc1,nc2,nc3,nc4,EN = x
    IcQ1,IbQ1 = npn(VB1, nc1, 0.0)
    IcQ2,IbQ2 = pnp(VB2P, nc2, VOUT)
    IcQ3,IbQ3 = npn(VB3, nc3, 0.0)
    IcQ4,IbQ4 = npn(VB3, nc4, 0.0)
    e = np.empty(8)
    e[0] = (VB1-VIN)/R1 + VB1/R2 + IbQ1
    e[1] = (nc1-VB3)/R4 + IcQ1
    e[2] = (VB3-VOUT)/R3 + (VB3-nc1)/R4 + (VB3-nc2)/R5 + VB3/R6 + IbQ3 + IbQ4
    e[3] = (nc2-VB3)/R5 + IcQ2
    e[4] = (VB2P-VOUT)/R11 + (VB2P-nc4)/R10 + IbQ2
    e[5] = (nc4-VB2P)/R10 + IcQ4
    e[6] = (nc3-EN)/R7 + IcQ3
    e[7] = (EN-VIN)/R8 + EN/R9 + (EN-nc3)/R7 + I_EN
    return e

def solve(VIN,VOUT,x0):
    x,_,ier,_ = fsolve(residuals, x0, args=(VIN,VOUT), full_output=True, xtol=1e-12)
    return x, ier

def trace(param, values, fixed, start):
    x = np.array(start, float); out = []
    for v in values:
        VIN = v if param=='VIN'  else fixed
        VOUT= v if param=='VOUT' else fixed
        x,_ = solve(VIN,VOUT,x); out.append((v, x.copy()))
    return out

LATCHED = lambda EN: EN < 0.4   # EN below shutdown floor

x_unlatched = [2.0,0.2,11.9,0.3,0.2,11.0,0.1,3.0]
x_latched   = [2.0,0.75,6.0,0.2,0.8,0.2,0.1,0.2]

In [ ]:
# --- SET: VOUT held at 12, sweep VIN down ---
down = trace('VIN', np.linspace(12,0.5,150), 12.0, x_unlatched)
set_vin = next((v for v,x in down if LATCHED(x[7])), None)

# --- HOLD: from the latched bottom, raise VIN to 22.5 (VOUT still 12) ---
up = trace('VIN', np.linspace(0.5,22.5,200), 12.0, down[-1][1])
en_max_hold = max(x[7] for v,x in up)
held = all(LATCHED(x[7]) for v,x in up if v > 6)

# --- RESET: continue from the genuinely-latched HOLD state, sweep VOUT down ---
# (seeding cold from x_latched at VIN=22.5 can fall into the unlatched basin,
#  so we start from up[-1], which we just confirmed is latched.)
rst = trace('VOUT', np.linspace(12,0.0,200), 22.5, up[-1][1])
reset_vout = None
prev = True
for v,x in rst:
    if prev and not LATCHED(x[7]):
        reset_vout = v; break
    prev = LATCHED(x[7])

print('DC sanity check (generic model):')
print(f'  SET   : latch engages near VIN ~ {set_vin:.2f} V   '
      f'(cf. IC falling {VIN_FALL:.2f} V)')
print(f'  HOLD  : VIN raised to 22.5 V, VOUT=12 -> {"HELD" if held else "DROPPED"}, '
      f'max EN = {en_max_hold:.2f} V  (re-enable needs {V_EN_OP} V)')
print(f'  RESET : releases near VOUT ~ {reset_vout:.2f} V   (target < {VOUT_RELEASE} V)')
print()
print('  Interpretation:')
print('   - HOLD margin is large and trustworthy: EN stays far below 1.22 V.')
print('   - SET is governed by R1/R2 and is robust qualitatively.')
print('   - RESET absolute value is model-sensitive; CONFIRM IN LTSPICE.')

## 6. Findings & action items

**What’s right**
- The overall structure — four regions (startup, SET, HOLD, RESET) — matches
  the circuit, and the regenerative HOLD loop works with big margin.
- With the datasheet UVLO equation, the **300k/100k divider lands turn-on at
  4.28 V**, neatly inside the [3.8 V, 4.5 V] target window. Good choice.

**Issues found (vs. the original notebook)**
1. **Designator mismatch (biggest).** Original called R1/R2 = 100k/300k the
   “EN/UVLO divider.” Schematic UVLO divider is **R8/R9 = 300k/100k**; R1/R2 =
   **5k/1k** is the VIN-sense divider. Original Q1–Q4 indices were also shuffled.
2. **Threshold equation missing the pin-current offset.** Using the datasheet
   eqn `v_in = v_en_op(1+R8/R9) − R8·i_pin` with I_STBY = 2 µA and ΔI_HYS = 3.15
   µA gives **rising 4.28 V / falling 3.34 V**, not the 4.88/3.94 from my first
   pass (which dropped the 0.60 V standby offset) or the 4.38 in the original.
3. **SET vs IC self-off — now OK with thin margin.** Realized SET engages ~3.5 V
   (Q1 edge ~3.9 V), both above the corrected IC falling trip 3.34 V, so the
   latch grabs first. But full-engage ~3.5 V is just under V_OFF_MIN (3.6 V);
   raise R2/(R1+R2) (e.g. 4.7k/1.2k) if you want guaranteed margin.
4. **Power budget FAIL.** `MAX_DIVIDER_UW = 500 µW`, but R8/R9 = 300k/100k draws
   ~1.27 mW at 22.5 V (over budget above ~14 V). **And it can’t be fixed by
   scaling the divider up:** to hit 500 µW you need ~1 MΩ total, but then the
   I_STBY+ΔI_HYS (5.15 µA) through a ~760k top resistor adds ~3.9 V of offset
   that collapses the thresholds. This simple-divider UVLO can be low-power **or**
   high-VIN-tolerant, not both — a topology decision is needed.
5. **RESET may release too high.** Generic model puts RESET near ~8–9 V vs the 2 V
   target; if real, residual VOUT energy could back-feed. **Top LTSpice check.**
   Likely fix: larger R4 or re-ratio R3/R5/R6.
6. **Incoherent constants (original notebook).** `PWR_LIMIT` 200e-3 vs comment
   “1000uW” vs 1000e-6 (200×); `V_USB_MAX` redefined per cell; copy-paste comment
   bug on the R5 line; `i_r8` computed twice; undefined names (`V_LATCH_ON_CEIL`).
   This notebook fixes those with one constants block + the real `constants.py`.

**Action items (in priority order)**
1. **Resolve the power-budget conflict** (finding 4): relax the 500 µW budget, cap
   the assumed max VIN, or switch the UVLO to an active / series-element approach.
2. Run the real LTSpice `.asc` and read the **RESET VOUT** and **SET VIN** trips.
3. If you want margin: raise R2/(R1+R2) for SET; raise R4 / re-ratio core for RESET.
4. Re-verify HOLD margin after any change (EN should stay ≪ 1.22 V at VIN = 22.5 V).